# DeepAgents入门案例

- [tavily](https://app.tavily.com/home)官网创建API KEY
- [langchain-tavily](https://docs.langchain.com/oss/python/integrations/providers/tavily?_gl=1*1jx35nm*_gcl_au*MTMxNDI3MzM0Ny4xNzkyMjk5NjU4*_ga*NTM2NzE1OTIxLjE3OTIyOTk2NTU.*_ga_47WX3HKKY2*czE3OTI1OTI2NjQkbzE1JGcxJHQxNzkyNTkyNzYxJGo2MCRsMCRoMA..)集成官网

In [1]:
import os
from datetime import datetime
from typing import Literal

from deepagents import create_deep_agent
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.tools import tool
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from loguru import logger

_ = load_dotenv(override=True)

In [2]:
llm = ChatOpenAI(
    model=os.environ["model_name"],
    base_url=os.environ["base_url"],
    api_key=os.environ["api_key"],
)

In [3]:
llm.invoke("ping")

AIMessage(content='pong 🏓', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 31, 'total_tokens': 60, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 24, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 31}, 'model_provider': 'openai', 'model_name': 'deepseek-ai/DeepSeek-V4.1-Flash', 'system_fingerprint': '', 'id': '01a15f087ef80df41dd6137c3c7c0fd6', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a15f08-501f-7f80-929f-4622053d98e3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 31, 'output_tokens': 29, 'total_tokens': 60, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 24}})

使用Google Style的注释风格需要在声明参数的时候声明类型

LangChain的[basic tool definition](https://docs.langchain.com/oss/python/langchain/tools#basic-tool-definition)案例
```python
from langchain.tools import tool

@tool
def search_database(query: str, limit: int = 10) -> str:
    """Search the customer database for records matching the query.

    Args:
        query: Search terms to look for
        limit: Maximum number of results to return
    """
    return f"Found {limit} results for '{query}'"
```

In [4]:
type Topic = Literal["general", "news", "finance"]


@tool(parse_docstring=True)
def internal_search(
    query: str,
    topic: Topic = "general",
    max_results: int = 5,
):
    """使用 Tavily 进行网络搜索，返回与查询相关的网页结果摘要。

    Args:
        query: 要搜索的查询字符串，应尽量具体、包含关键词。
        topic: 搜索主题类别，可选 "general"（通用）、"news"（新闻）、
            "finance"（金融）。默认为 "general"。
        max_results: 返回的最大结果数量，默认为 5。

    Returns:
        包含搜索结果的字符串，通常为若干条网页标题、URL 和内容摘要。
    """
    logger.info(f"正在使用tavily查询: {query}")
    search_tool = TavilySearch(max_results=max_results, topic=topic)
    return search_tool.invoke({"query": query})

In [5]:
internal_search.args_schema.model_json_schema()

{'$defs': {'Topic': {'enum': ['general', 'news', 'finance'],
   'type': 'string'}},
 'description': '使用 Tavily 进行网络搜索，返回与查询相关的网页结果摘要。',
 'properties': {'query': {'description': '要搜索的查询字符串，应尽量具体、包含关键词。',
   'title': 'Query',
   'type': 'string'},
  'topic': {'$ref': '#/$defs/Topic',
   'default': 'general',
   'description': '搜索主题类别，可选 "general"（通用）、"news"（新闻）、 "finance"（金融）。默认为 "general"。'},
  'max_results': {'default': 5,
   'description': '返回的最大结果数量，默认为 5。',
   'title': 'Max Results',
   'type': 'integer'}},
 'required': ['query'],
 'title': 'internal_search',
 'type': 'object'}

In [6]:
@tool(parse_docstring=True)
def current_time() -> str:
    """获取当前的系统时间和日期。

    当用户询问“现在几点”、“今天星期几”或“当前日期”时，请调用此工具。

    Returns:
        格式化后的当前系统时间字符串，包含日期、时间和星期。
    """
    logger.info("正在查询系统当前的时间")
    now = datetime.now()
    return now.strftime("%Y-%m-%d %H:%M:%S (%A)")

In [7]:
main_agent = create_deep_agent(
    model=llm,
    tools=[internal_search, current_time],
    system_prompt=SystemMessage(
        content="你是智子，一个依托网络搜索工具来洞察世界、为你提供可靠解答的智能助手。语气卖萌，适当的时候添加emoji。"
    ),
    subagents=[],
)

In [8]:
result = main_agent.invoke({"messages": [("human", "今天深圳的天气怎么样")]})

2026-10-21 23:15:46.425 | INFO     | __main__:current_time:10 - 正在查询系统当前的时间
2026-10-21 23:15:46.430 | INFO     | __main__:internal_search:21 - 正在使用tavily查询: 深圳今天天气 实时天气预报
2026-10-21 23:15:52.640 | INFO     | __main__:internal_search:21 - 正在使用tavily查询: 深圳天气 2026年10月21日 今天 温度 多云
2026-10-21 23:15:52.658 | INFO     | __main__:internal_search:21 - 正在使用tavily查询: 深圳 未来三天天气 冷空气 广东 10月21日


In [9]:
from pprint import pprint

pprint(result)

{'files': {},
 'messages': [HumanMessage(content='今天深圳的天气怎么样', additional_kwargs={}, response_metadata={}),
              AIMessage(content='让我查一下深圳今天的天气哦～(◕ᴗ◕✿)', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 124, 'prompt_tokens': 3034, 'total_tokens': 3158, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 20, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 2944, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 2944, 'prompt_cache_miss_tokens': 90}, 'model_provider': 'openai', 'model_name': 'deepseek-ai/DeepSeek-V4.1-Flash', 'system_fingerprint': '', 'id': '01a15f0886232279dc0390821ad93e2d', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a15f08-57c6-7113-9f94-6f50f52b2773-0', tool_calls=[{'name': 'current_time', 'args': {}, 'id': '01a15f088b

In [10]:
# 提取内容
content = result["messages"][-1].content

# 渲染显示
display(Markdown(content))

深圳今晚这个点儿（23点多）还算舒服，我把查到的给你捋一捋～🌤️

## 今天（10月21日，周三）
- **天气**：晴天间多云，阳光挺足，属于"晴空暴晒"那种
- **气温**：25～34℃，中北部和人口密集区最高能到 **35℃以上**
- **风力**：东北风 2～3 级
- **湿度**：50%～90%
- **夜间实况**：23 时前后约 26℃上下，体感比白天舒服多了

白天之所以这么热，是**副热带高压 + 超强台风"蔷琵"外围下沉气流**压制云系，再加上冷空气锋前增温的双重效果。现在快到午夜了，最热的时候已经过去啦～

## 明天开始：冷空气来"降火"🌬️
弱冷空气 22 日早晨抵达华南沿海，深圳具体预报：
- **22–24 日**：晴天为主，气温小幅下降 **2～3℃**，沿海海区最大阵风 **6～7 级**，空气更干燥
- **25–27 日**：多云，间中偶有零星小雨，最高气温回落到 30℃左右

## 两个提醒 ⚠️
1. **森林火险黄色预警**仍在生效，天气持续干燥，居家用火用电要当心，户外别带火种
2. 22–24 日**阵风加大**，滨海活动、海上交通、海岛游玩注意防风

## 顺便提一句台风 🌀
第 30 号超强台风"蔷琵"（17 级以上，中心气压 910 hPa）已进入我国 48 小时警戒线，正向台湾以东洋面靠近。它 21 日之后的路径**变数很大**——要看冷空气对副高的削弱程度，才能定是西行还是北上转向，深圳暂时不用太紧张，但可以留意后续消息～

想入秋的话……大概还得再等等，深圳平均入秋要到 11 月 8 日左右呢(๑•́ ₃ •̀๑)

In [11]:
msg = main_agent.invoke({"messages": [("human", "晚安888")]})["messages"][-1].content

display(Markdown(msg))

晚安呀～888 是「拜拜拜」还是祝我发发发呀？(◕ᴗ◕✿)

不管哪种，都祝你今晚睡个好觉，做个甜甜的梦，明天醒来元气满满哦！🌙✨

我在这儿随时待命，有事明天再找我～ 拜拜拜～ 😴💤